In [31]:
import os

import pandas as pd
import numpy as np

import functions.cnn_helpers as help_funcs

# DNA MERFISH data (Su et al. 2020)

## Read distance info and save as arrays

In [25]:
dist_data = pd.read_csv('../data/Su_2020/selected_structures_95_1000_nm_thr.tsv', sep='\t')
transc_data = pd.read_csv('../data/Su_2020/transcriptional_data_95_1000_nm_thr.tsv', sep='\t')

dataset = pd.merge(dist_data, transc_data.iloc[:, :2], how='left', on='Chromosome copy number')

n_loci = 38

### Iterate through all structures and save them as an image

In [28]:
for id, row in dataset.iterrows():
    
    struct_id = int(row['Chromosome copy number'])
    state = int(row['BACH1'])
    if state == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    dists = row.iloc[1 : -1]
    
    locus_i = 0
    dist_i = np.zeros((n_loci, n_loci))
    
    for i in range(0, n_loci):
        for j in range(i + 1, n_loci):
            dist_i[i,j] = dists.iloc[locus_i]
            
            locus_i += 1
    
    dist_i = dist_i + dist_i.T
    
    
    dist_i = pd.DataFrame(dist_i)
    dist_i.to_csv(f"../data/Su_2020/distance_maps_95_1000/{state_save}/structure_{struct_id}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [29]:
help_funcs.kfold_splits(src_folder = '../data/Su_2020/distance_maps_95_1000/', dst_folder = '../data/Su_2020/CNN_splits_balanced_95_1000')

Creating fold 1/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 2/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 3/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 4/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 5/5...
Train size: 1556, Validation size: 638, Test size: 638
Created 5 stratified splits under '../data/Su_2020/CNN_splits_balanced_95_1000'.


In [5]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


# DNA MERFISH data (Su et al. 2020) - simulated dataset

## Read matrices, fill missing triangles and convert from Amber A.U. to real physical nanometers

In [55]:
transc_data = pd.read_csv('../data/Su_2020/simulated_dataset/structures_transcription_data.tsv', sep='\t')
#transc_data = transc_data[['Transcription', 'structure']]

In [56]:
transc_data

,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,2-3,...,7-8,7-9,7-10,8-9,8-10,9-10,Chromosome copy number,Transcription,dist_index,structure
0,7.0,7.0,7.0,22.0,37.0,7.0,22.0,7.0,7.0,7.0,...,22.0,22.0,7.0,22.0,7.0,22.0,1091,0,1,0
1,7.0,22.0,127.0,127.0,112.0,232.0,157.0,187.0,217.0,22.0,...,97.0,52.0,67.0,37.0,112.0,82.0,7454,1,2,813
2,7.0,37.0,97.0,52.0,52.0,142.0,157.0,142.0,217.0,37.0,...,22.0,157.0,82.0,172.0,67.0,202.0,7780,0,3,1412
3,7.0,37.0,157.0,97.0,127.0,112.0,67.0,127.0,112.0,37.0,...,97.0,82.0,202.0,67.0,187.0,232.0,1654,0,4,1772
4,7.0,52.0,82.0,67.0,157.0,187.0,157.0,277.0,202.0,37.0,...,67.0,157.0,67.0,157.0,82.0,82.0,10368,1,5,2493
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3010,322.0,337.0,352.0,202.0,217.0,112.0,37.0,52.0,7.0,52.0,...,127.0,82.0,97.0,97.0,52.0,52.0,11786,0,2999,1499436
3011,337.0,127.0,37.0,142.0,67.0,142.0,232.0,187.0,247.0,367.0,...,142.0,97.0,112.0,247.0,187.0,112.0,9630,1,3000,1499987
3012,337.0,142.0,172.0,187.0,202.0,202.0,187.0,322.0,202.0,202.0,...,157.0,187.0,202.0,172.0,67.0,142.0,3224,0,3001,1500090
3013,367.0,262.0,247.0,112.0,247.0,217.0,112.0,127.0,172.0,142.0,...,112.0,82.0,187.0,52.0,82.0,127.0,8853,1,3002,1500772


In [57]:
transc_data.head()

,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,2-3,...,7-8,7-9,7-10,8-9,8-10,9-10,Chromosome copy number,Transcription,dist_index,structure
0,7.0,7.0,7.0,22.0,37.0,7.0,22.0,7.0,7.0,7.0,...,22.0,22.0,7.0,22.0,7.0,22.0,1091,0,1,0
1,7.0,22.0,127.0,127.0,112.0,232.0,157.0,187.0,217.0,22.0,...,97.0,52.0,67.0,37.0,112.0,82.0,7454,1,2,813
2,7.0,37.0,97.0,52.0,52.0,142.0,157.0,142.0,217.0,37.0,...,22.0,157.0,82.0,172.0,67.0,202.0,7780,0,3,1412
3,7.0,37.0,157.0,97.0,127.0,112.0,67.0,127.0,112.0,37.0,...,97.0,82.0,202.0,67.0,187.0,232.0,1654,0,4,1772
4,7.0,52.0,82.0,67.0,157.0,187.0,157.0,277.0,202.0,37.0,...,67.0,157.0,67.0,157.0,82.0,82.0,10368,1,5,2493


In [59]:
original_mats_folder = '../data/Su_2020/simulated_dataset/matrices_AU_upper_diagonal'

dist_mats = os.listdir(original_mats_folder)

In [62]:
# Go through each matrix, process and save according to transcriptional state
scaling_factor = 0.0237699544

for mat_i_file in dist_mats:
    mat_i = pd.read_csv(f'{original_mats_folder}/{mat_i_file}', sep='\t', header=None).to_numpy() / scaling_factor / 10
    
    mat_i[np.isnan(mat_i)] = 0
    mat_i = mat_i + np.transpose(mat_i)

    frame = int(mat_i_file.split('_')[-1].split('.')[0])
    frame_transc = transc_data[transc_data['structure'] == frame]['Transcription'].values[0]
     
    if frame_transc == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    mat_i = pd.DataFrame(mat_i)
    mat_i.to_csv(f"../data/Su_2020/simulated_dataset/matrices_transformed_classified/{state_save}/structure_{frame}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [63]:
help_funcs.kfold_splits(src_folder = '../data/Su_2020/simulated_dataset/matrices_transformed_classified/', dst_folder = '../data/Su_2020/simulated_dataset/CNN_splits_balanced_95_1000')

Creating fold 1/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 2/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 3/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 4/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 5/5...
Train size: 1466, Validation size: 603, Test size: 603
Created 5 stratified splits under '../data/Su_2020/simulated_dataset/CNN_splits_balanced_95_1000'.


In [66]:
len(os.listdir('../data/Su_2020/simulated_dataset/CNN_splits_balanced_95_1000_zscore/fold_4/test/inactive'))

359

In [119]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


In [120]:
samples

[('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_1248573.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_666262.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_846540.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_23567.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_314629.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_995297.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_772001.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_625011.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/str

# ORCA data (Mateo et al. 2020)

## Read distance info and save as arrays

In [42]:
dist_data = pd.read_csv('../data/Mateo_2019/selected_structures_75_5000_filtered.tsv', sep='\t')
transc_data = pd.read_csv('../data/Mateo_2019/transcriptional_data_75_5000_filtered.tsv', sep='\t')

dataset = pd.merge(dist_data, transc_data.iloc[:, :2], how='left', on='cellNumber')

n_loci = 52

# Shuffle labels to test performance
labels = np.array(dataset['Abd-A_Intron'])
labels = np.random.permutation(labels)
dataset['Abd-A_Intron'] = labels

In [44]:
dataset['Abd-A_Intron'].value_counts()

Abd-A_Intron
0    1641
1     653
Name: count, dtype: int64

### Iterate through all structures and save them as an image

In [46]:
for id, row in dataset.iterrows():
    
    struct_id = int(row['cellNumber'])
    state = int(row['Abd-A_Intron'])
    if state == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    dists = row.iloc[1 : -1]
    
    locus_i = 0
    dist_i = np.zeros((n_loci, n_loci))
    
    for i in range(0, n_loci):
        for j in range(i + 1, n_loci):
            dist_i[i,j] = dists.iloc[locus_i]
            
            locus_i += 1
    
    dist_i = dist_i + dist_i.T
    
    
    dist_i = pd.DataFrame(dist_i)
    dist_i.to_csv(f"../data/Mateo_2019/distance_maps_75_5000_filtered_full_shuffled/{state_save}/structure_{struct_id}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [47]:
help_funcs.final_split(src_folder = '../data/Mateo_2019/distance_maps_75_5000_filtered_full_shuffled/', dst_folder = '../data/Mateo_2019/CNN_splits_balanced_75_5000_filtered_full_shuffled', test_size=0.2)

Train size: 1044, Test size: 459
Created splits under '../data/Mateo_2019/CNN_splits_balanced_75_5000_filtered_full_shuffled'.


In [ ]:
# Also try with unbalanced dataset
#help_funcs.kfold_splits_unbalanced(src_folder = '../data/Mateo_2019/distance_maps_75_5000_filtered/', dst_folder = '../data/Mateo_2019/CNN_splits_unbalanced_75_5000_filtered', test_size=0.2, val_size=0.25)

Creating fold 1/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 2/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 3/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 4/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 5/5...
Train size: 1376, Validation size: 459, Test size: 459
Created 5 stratified splits under '../data/Mateo_2019/CNN_splits_unbalanced_75_5000_filtered'.


In [22]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


In [23]:
samples

[('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_24585.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_12751.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_45609.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_10146.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_27926.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_33953.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_49741.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_17615.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/in

# ORCA MERFISH data (Mateo et al. 2019) - simulated dataset

## Read matrices, fill missing triangles and convert from Amber A.U. to real physical nanometers

In [25]:
transc_data = pd.read_csv('../data/Mateo_2019/simulated_dataset/structures_transcription_data.tsv', sep='\t')
#transc_data = transc_data[['Transcription', 'structure']]

In [26]:
transc_data.head()

,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,2-3,...,7-8,7-9,7-10,8-9,8-10,9-10,cellNumber,Transcription,dist_index,structure
0,2,2,2,6,10,2,6,10,2,6,...,6,10,6,6,6,10,31299.0,1,1,0
1,2,2,6,14,14,18,22,10,6,2,...,6,10,14,14,22,6,19306.0,0,2,528
2,2,6,6,10,10,6,10,6,10,6,...,14,6,14,10,10,14,2780.0,0,3,1001
3,2,6,6,10,14,10,10,14,14,6,...,10,10,10,10,10,2,32380.0,0,4,1500
4,2,6,6,26,18,30,50,34,30,10,...,30,10,14,22,22,10,32290.0,1,5,2147


In [27]:
original_mats_folder = '../data/Mateo_2019/simulated_dataset/matrices_AU_upper_diagonal'

dist_mats = os.listdir(original_mats_folder)

In [28]:
dist_mats

['frame_1912_910254.tsv',
 'frame_1403_668181.tsv',
 'frame_140_66568.tsv',
 'frame_1962_932006.tsv',
 'frame_1717_817545.tsv',
 'frame_1393_663000.tsv',
 'frame_1242_592056.tsv',
 'frame_1521_726123.tsv',
 'frame_463_220077.tsv',
 'frame_1979_940031.tsv',
 'frame_982_469600.tsv',
 'frame_2082_986471.tsv',
 'frame_1882_896141.tsv',
 'frame_1599_764056.tsv',
 'frame_1873_891838.tsv',
 'frame_651_311511.tsv',
 'frame_1167_556548.tsv',
 'frame_1418_675940.tsv',
 'frame_509_242022.tsv',
 'frame_1034_493817.tsv',
 'frame_1518_724659.tsv',
 'frame_1786_850503.tsv',
 'frame_276_130893.tsv',
 'frame_1159_552618.tsv',
 'frame_1891_900584.tsv',
 'frame_1949_926270.tsv',
 'frame_1399_666120.tsv',
 'frame_278_131000.tsv',
 'frame_1388_660553.tsv',
 'frame_774_372042.tsv',
 'frame_1710_814163.tsv',
 'frame_548_261824.tsv',
 'frame_978_468484.tsv',
 'frame_566_270508.tsv',
 'frame_824_396587.tsv',
 'frame_1711_814523.tsv',
 'frame_1236_589017.tsv',
 'frame_953_455880.tsv',
 'frame_1549_739503.tsv',


In [29]:
transc_data

,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,2-3,...,7-8,7-9,7-10,8-9,8-10,9-10,cellNumber,Transcription,dist_index,structure
0,2,2,2,6,10,2,6,10,2,6,...,6,10,6,6,6,10,31299.0,1,1,0
1,2,2,6,14,14,18,22,10,6,2,...,6,10,14,14,22,6,19306.0,0,2,528
2,2,6,6,10,10,6,10,6,10,6,...,14,6,14,10,10,14,2780.0,0,3,1001
3,2,6,6,10,14,10,10,14,14,6,...,10,10,10,10,10,2,32380.0,0,4,1500
4,2,6,6,26,18,30,50,34,30,10,...,30,10,14,22,22,10,32290.0,1,5,2147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2185,62,42,26,34,30,38,22,34,38,26,...,14,10,10,14,22,10,49524.0,0,2065,1032147
2186,62,54,50,50,46,58,58,54,62,18,...,6,10,6,6,6,10,17049.0,1,2066,1032623
2187,66,62,62,62,66,58,70,74,82,6,...,18,18,26,10,14,10,44596.0,0,2067,1033050
2188,66,62,62,62,66,58,70,74,82,6,...,18,18,26,10,14,10,44596.0,0,2067,1033039


In [9]:
# Go through each matrix, process and save according to transcriptional state
scaling_factor = 0.0051234457

for mat_i_file in dist_mats:
    mat_i = pd.read_csv(f'{original_mats_folder}/{mat_i_file}', sep='\t', header=None).to_numpy() / scaling_factor / 10
    
    mat_i[np.isnan(mat_i)] = 0
    mat_i = mat_i + np.transpose(mat_i)

    frame = int(mat_i_file.split('_')[-1].split('.')[0])
    frame_transc = transc_data[transc_data['structure'] == frame]['Transcription'].values[0]
     
    if frame_transc == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    mat_i = pd.DataFrame(mat_i)
    mat_i.to_csv(f"../data/Mateo_2019/simulated_dataset/matrices_transformed_classified/{state_save}/structure_{frame}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [10]:
help_funcs.kfold_splits(src_folder = '../data/Mateo_2019/simulated_dataset/matrices_transformed_classified/', dst_folder = '../data/Mateo_2019/simulated_dataset/CNN_splits_balanced_75_5000')

Creating fold 1/5...
Train size: 754, Validation size: 438, Test size: 438
Creating fold 2/5...
Train size: 754, Validation size: 438, Test size: 438
Creating fold 3/5...
Train size: 754, Validation size: 438, Test size: 438
Creating fold 4/5...
Train size: 754, Validation size: 438, Test size: 438
Creating fold 5/5...
Train size: 754, Validation size: 438, Test size: 438
Created 5 stratified splits under '../data/Mateo_2019/simulated_dataset/CNN_splits_balanced_75_5000'.


#### Final evaluation with test/train set

In [30]:
# Redo, only dividing into test and training sets
help_funcs.final_split(src_folder = '../data/Mateo_2019/simulated_dataset/matrices_transformed_classified/', dst_folder = '../data/Mateo_2019/simulated_dataset/CNN_splits_balanced_75_5000_full')

Train size: 1006, Test size: 438
Created splits under '../data/Mateo_2019/simulated_dataset/CNN_splits_balanced_75_5000_full'.
